<a id="top"></a>

# Lab L0903: Write your first eval set

```yaml
title: "Lab L0903: Write your first eval set"
keywords: evaluation, eval case, scorer, runner, outcome, trajectory, regression case, brittleness, lab
estimated duration: 35
```

> **Lesson:** L09. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L09/objectives.md).
> Runs offline — no API key needed (scripted `FakeModel`).

**Learning objectives.** By the end of this lab you can:

- **write an outcome scorer and a trajectory scorer** — score the *answer* and the *path*;
- **turn an L08 failure into a regression case** — a check that fails when the bug is present and passes when it's fixed;
- **recognize and loosen a brittle check** — use the loosest check that still catches the bug.

> Solutions: `L0903_lab_solutions.ipynb`. Pure Python — no API key needed.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — an outcome scorer](#2-problem-1--an-outcome-scorer)
- [3. Problem 2 — a trajectory scorer](#3-problem-2--a-trajectory-scorer)
- [4. Problem 3 — a regression case from an L08 failure](#4-problem-3--a-regression-case-from-an-l08-failure)
- [5. Problem 4 — loosen a brittle check](#5-problem-4--loosen-a-brittle-check)
- [6. Self-check](#6-self-check)

## 1. Setup

Run this cell once. It imports the frozen `common/` library and a `run_case` that drives the loop with a scripted `FakeModel` — deterministic, no API key. You write **cases** and **scorers**; `run_case` produces the runs.

In [1]:
from fluffy_potato_curriculum.common.agent_loop import RunResult, run
from fluffy_potato_curriculum.common.evals import (
    EvalCase,
    EvalResult,
    evaluate,
    tool_calls,
    tool_trajectory,
)
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    FakeResponse,
    response,
    text_block,
    tool_use_block,
)
from fluffy_potato_curriculum.common.tools import TOOLS

# case id -> the model's scripted moves. FakeModel repeats its LAST line (a runaway).
SCRIPTS: dict[str, list[FakeResponse]] = {
    "tokyo": [
        response([tool_use_block("c1", "calculator", {"expression": "17*23"})]),
        response([tool_use_block("c2", "lookup", {"city": "Tokyo"})]),
        response([text_block("17 * 23 is 391, and Tokyo has a population of 37000000.")]),
    ],
    "tokyo_reworded": [  # correct, but written with commas
        response([tool_use_block("c1", "lookup", {"city": "Tokyo"})]),
        response([text_block("About 37,000,000 people live in Tokyo.")]),
    ],
    "atlantis_runaway": [  # one line -> repeats -> max_steps
        response([tool_use_block("c1", "lookup", {"city": "Atlantis"})]),
    ],
    "atlantis_fixed": [  # gives up cleanly -> natural termination
        response([text_block("I do not have a population on file for Atlantis.")]),
    ],
}


def run_case(case: EvalCase) -> RunResult:
    """Run the loop for one case with its scripted FakeModel (fresh each call)."""
    model = FakeModel(SCRIPTS[case.id])
    return run(model, TOOLS, case.inputs["task"], max_steps=case.inputs.get("max_steps", 8))


print("scripts ready:", ", ".join(SCRIPTS))

scripts ready: tokyo, tokyo_reworded, atlantis_runaway, atlantis_fixed


[↑ Back to top](#top)

## 2. Problem 1 — an outcome scorer

Write `answer_correct`: an **outcome** scorer that reads `run.final_text` and returns an `EvalResult` whose `score` is `True` when the reference answer (`example.reference_outputs["answer"]`) appears in the final text. Use `key="answer_correct"`.

The asserts run it through the harness on the `tokyo` case (which should pass).

In [2]:
def answer_correct(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Outcome scorer: is the reference answer a substring of the final text?"""
    expected = str(example.reference_outputs["answer"])
    return EvalResult(key="answer_correct", score=expected in run.final_text)


tokyo_case = EvalCase(
    id="tokyo",
    inputs={"task": "What is 17 * 23, and the population of Tokyo?"},
    reference_outputs={"answer": "37000000"},
)
report = evaluate(run_case, [tokyo_case], [answer_correct])
print(report.table())
assert report.pass_rate("tokyo", "answer_correct") == 1.0

case   answer_correct  ALL
tokyo  1/1             1/1


[↑ Back to top](#top)

## 3. Problem 2 — a trajectory scorer

For an agent, the *path* matters too. Write `expected_tools`: a **trajectory** scorer that uses `tool_trajectory(run)` (the ordered tool-call names) and returns `score=True` when the path equals `example.reference_outputs["expected_tools"]`. Use `key="expected_tools"`.

The `tokyo` case should call `calculator` then `lookup`.

In [3]:
def expected_tools(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Trajectory scorer: did the ordered tool calls match the reference path?"""
    expected = list(example.reference_outputs["expected_tools"])
    return EvalResult(key="expected_tools", score=tool_trajectory(run) == expected)


tokyo_traj_case = EvalCase(
    id="tokyo",
    inputs={"task": "What is 17 * 23, and the population of Tokyo?"},
    reference_outputs={"expected_tools": ["calculator", "lookup"]},
)
report = evaluate(run_case, [tokyo_traj_case], [expected_tools])
print(report.table())
assert report.pass_rate("tokyo", "expected_tools") == 1.0

case   expected_tools  ALL
tokyo  1/1             1/1


[↑ Back to top](#top)

## 4. Problem 3 — a regression case from an L08 failure

The headline rule: **trace a failure → write a case that catches it → keep it forever.** In L08 you saw a **runaway**: the same `(tool, args)` repeating, ending in `max_steps`. Turn it into a scorer + a case.

Write `no_runaway`: a trajectory scorer that fails when any `(name, args)` pair repeats in `tool_calls(run)` **or** the run terminated `"max_steps"`. A good regression check is **red when broken, green when fixed** — so it must fail on `atlantis_runaway` and pass on `atlantis_fixed`.

In [4]:
def no_runaway(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Trajectory scorer: no repeated (tool, args) call, and didn't hit the step cap."""
    signatures = [(name, tuple(sorted(args.items()))) for name, args in tool_calls(run)]
    repeated = len(signatures) != len(set(signatures))
    hit_cap = run.termination == "max_steps"
    return EvalResult(key="no_runaway", score=not (repeated or hit_cap))


runaway_case = EvalCase(
    id="atlantis_runaway", inputs={"task": "Population of Atlantis?", "max_steps": 4}
)
fixed_case = EvalCase(id="atlantis_fixed", inputs={"task": "Population of Atlantis?"})
report = evaluate(run_case, [runaway_case, fixed_case], [no_runaway])
print(report.table())
# Red when broken, green when fixed:
assert report.pass_rate("atlantis_runaway", "no_runaway") == 0.0
assert report.pass_rate("atlantis_fixed", "no_runaway") == 1.0

case              no_runaway  ALL
atlantis_runaway  0/1         0/1
atlantis_fixed    1/1         1/1


[↑ Back to top](#top)

## 5. Problem 4 — loosen a brittle check

The `tokyo_reworded` run is **correct** but writes `37,000,000` with commas — so your `answer_correct` (a plain substring) fails it. A red on a correct run is worse than no check: it trains you to ignore reds.

Write `answer_correct_loose`: same idea, but normalize **commas and spaces** out of both strings before comparing. It should turn `tokyo_reworded` green. Keep `key="answer_correct"`.

In [5]:
reworded_case = EvalCase(
    id="tokyo_reworded",
    inputs={"task": "What is the population of Tokyo?"},
    reference_outputs={"answer": "37000000"},
)
print("strict check (brittle):")
print(evaluate(run_case, [reworded_case], [answer_correct]).table())


def answer_correct_loose(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Outcome scorer, loosened: compare with commas and spaces normalized out."""

    def norm(text: str) -> str:
        return text.replace(",", "").replace(" ", "")

    expected = str(example.reference_outputs["answer"])
    return EvalResult(key="answer_correct", score=norm(expected) in norm(run.final_text))


print("\nloosened check:")
report = evaluate(run_case, [reworded_case], [answer_correct_loose])
print(report.table())
assert report.pass_rate("tokyo_reworded", "answer_correct") == 1.0

strict check (brittle):
case            answer_correct  ALL
tokyo_reworded  0/1             0/1

loosened check:
case            answer_correct  ALL
tokyo_reworded  1/1             1/1


**TODO (written):** Name one quality of an answer that *no* string-normalizing check could score — something you'd need an LLM-as-judge (or a human) for. Edit this cell with your answer.

_TODO_

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0903_lab_solutions.ipynb`. Done when:

- **Problem 1** `answer_correct` passes the `tokyo` case (`1/1`).
- **Problem 2** `expected_tools` passes the `tokyo` trajectory case (`1/1`).
- **Problem 3** `no_runaway` is **red on `atlantis_runaway`** (`0/1`) and **green on `atlantis_fixed`** (`1/1`) — a real regression case.
- **Problem 4** the strict check fails `tokyo_reworded`, your `answer_correct_loose` turns it green, and your written answer names a quality only a judge/human can score (e.g. tone, "did it acknowledge the failure gracefully", factual correctness beyond the reference).

You can now write outcome and trajectory scorers and grow a regression case from a real failure. In **L0905** you'll run an eval set many times and read **pass rates**.

[↑ Back to top](#top)